In [34]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
import pandas as pd

df = pd.read_csv('../data/silver/cleaned_imdb.csv')

In [ ]:
analyzer = SentimentIntensityAnalyzer()

vader_score = df["review"].apply(analyzer.polarity_scores)

#  vader is a series of dicts --> we convert it to dataframe



(50000, 4)

In [35]:
vader_df = pd.DataFrame(list(vader_score))

X_vader = vader_df[["neg", "neu", "pos", "compound"]]

X_vader.shape


(50000, 4)

In [36]:
from textblob import TextBlob

df["tb_polarity"] = df["review"].apply(lambda x : TextBlob(x).sentiment.polarity)
df["tb_subjectivity"] = df["review"].apply(lambda x : TextBlob(x).sentiment.subjectivity)


In [37]:

X_tb= df[["tb_polarity", "tb_subjectivity"]]
X_tb.shape

(50000, 2)

In [38]:
df = df.drop(columns=["Unnamed: 0"], errors="ignore")
df = pd.concat([df, vader_df], axis=1)

In [40]:
df.head()

,review,sentiment,n_words,n_chars,tb_polarity,tb_subjectivity,neg,neu,pos,compound
0,one of the other reviewers has mentioned that ...,positive,307,1761,0.023433,0.490369,0.181,0.754,0.065,-0.9916
1,a wonderful little production. the filming tec...,positive,162,998,0.109722,0.559343,0.053,0.765,0.182,0.9670
2,i thought this was a wonderful way to spend ti...,positive,166,926,0.360960,0.655933,0.113,0.665,0.222,0.9745
3,basically there's a family where a little boy ...,negative,138,748,0.004167,0.459259,0.127,0.813,0.060,-0.9213
4,"petter mattei's ""love in the time of money"" is...",positive,230,1317,0.214483,0.451359,0.051,0.796,0.153,0.9766


In [48]:
df = df[["review", "sentiment","neg","neu","pos","compound","tb_polarity","tb_subjectivity"]]
df.head()

,review,sentiment,neg,neu,pos,compound,tb_polarity,tb_subjectivity
0,one of the other reviewers has mentioned that ...,positive,0.181,0.754,0.065,-0.9916,0.023433,0.490369
1,a wonderful little production. the filming tec...,positive,0.053,0.765,0.182,0.9670,0.109722,0.559343
2,i thought this was a wonderful way to spend ti...,positive,0.113,0.665,0.222,0.9745,0.360960,0.655933
3,basically there's a family where a little boy ...,negative,0.127,0.813,0.060,-0.9213,0.004167,0.459259
4,"petter mattei's ""love in the time of money"" is...",positive,0.051,0.796,0.153,0.9766,0.214483,0.451359


In [45]:
print("X_vader:", X_vader.shape)   # (50000, 4)
print("X_tb:", X_tb.shape)         # (50000, 2)

X_hybrid = pd.concat([X_vader, X_tb], axis=1)
print("X_hybrid:", X_hybrid.shape) # (50000, 6)


X_vader: (50000, 4)
X_tb: (50000, 2)
X_hybrid: (50000, 6)


In [46]:
y = (df["sentiment"] == "positive").astype(int)
print(y.value_counts())
print(y.value_counts(normalize=True))


sentiment
1    25000
0    25000
Name: count, dtype: int64
sentiment
1    0.5
0    0.5
Name: proportion, dtype: float64


In [47]:
print(X_vader.agg(["min","max"]).T)
print(X_tb.agg(["min","max"]).T)
print(X_hybrid.agg(["min","max"]).T)


             min     max
neg       0.0000  0.5770
neu       0.3390  1.0000
pos       0.0000  0.5230
compound -0.9997  0.9999
                 min  max
tb_polarity     -1.0  1.0
tb_subjectivity  0.0  1.0
                    min     max
neg              0.0000  0.5770
neu              0.3390  1.0000
pos              0.0000  0.5230
compound        -0.9997  0.9999
tb_polarity     -1.0000  1.0000
tb_subjectivity  0.0000  1.0000


In [50]:
df.to_csv("../data/gold/final_imdb.csv")